# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [74]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE


SEED = 32

In [3]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

.1

In [27]:
counts = fraud['fraud'].value_counts()

difference = abs(counts.iloc[0] - counts.iloc[1])
percentage = difference / counts.iloc[0] * 100

print(f"Diference: {percentage:.2f}%, We are dealing with imbalanced data")

Diference: 90.42%, We are dealing with imbalanced data


.2

In [82]:
# Select the features (X) and the target (y)
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y)  # Keeps the same proportion of the 3 species in both sets

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (800000, 7)
X_test shape:  (200000, 7)


In [83]:
# 1. Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 2. Create the Logistic Regression model
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)


# 3. Train the model
log_reg.fit(X_train_scaled, y_train)

print ("Model trained!")

Model trained!


In [84]:
# 4. Predictions
y_pred_train = log_reg.predict(X_train_scaled)
y_pred_test = log_reg.predict(X_test_scaled)


# 5. Accuracy
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy (Train): {acc_train * 100:.2f}%")
print(f"Accuracy (Test): {acc_test * 100:.2f}%")


# 6. Classification Report
print("\nClassification Report (Test):")
print(classification_report(
    y_test,
    y_pred_test,
    zero_division=0
))

Accuracy (Train): 95.90%
Accuracy (Test): 95.79%

Classification Report (Test):
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.88      0.60      0.71     17481

    accuracy                           0.96    200000
   macro avg       0.92      0.79      0.84    200000
weighted avg       0.96      0.96      0.95    200000



In [87]:
# 4. Predictions
y_pred_train = log_reg.predict(X_train_scaled)
y_pred_test = log_reg.predict(X_test_scaled)


# 5. Metrics
acc_train = accuracy_score(
    y_train,
    y_pred_train
)

acc_test = accuracy_score(
    y_test,
    y_pred_test
)

precision = precision_score(
    y_test,
    y_pred_test
)

recall = recall_score(
    y_test,
    y_pred_test
)

f1 = f1_score(
    y_test,
    y_pred_test
)

gap = acc_train - acc_test
print(f"Accuracy (Train): {acc_train * 100:.2f}%")
print(f"Accuracy (Test): {acc_test * 100:.2f}%")
print(f"Precision (Test): {precision * 100:.2f}%")
print(f"Recall (Test): {recall * 100:.2f}%")
print(f"F1-score (Test): {f1 * 100:.2f}%")
print(f"Train-Test Gap: {gap * 100:.2f}%")

Accuracy (Train): 95.90%
Accuracy (Test): 95.79%
Precision (Test): 88.47%
Recall (Test): 59.63%
F1-score (Test): 71.24%
Train-Test Gap: 0.10%


.3

The model need to find more true information because there is a 60% on recall making the model unreliable to detect frauds.

# .4 OVER SAMPLING

In [37]:
ros = RandomOverSampler(random_state=SEED)

X_train_over, y_train_over = ros.fit_resample(
    X_train,
    y_train
)
print("Original:")
print(y_train.value_counts())

print("\nAfter Oversampling:")
print(y_train_over.value_counts())

Original:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After Oversampling:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [40]:
# 2. Create the Logistic Regression model

log_reg_over = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)

log_reg_over.fit(X_train_over, y_train_over)

# 3. Train the model
log_reg_over.fit(X_train_over, y_train_over)


print ("Model trained!")

Model trained!


In [45]:
# 4. Predictions
y_pred_train_over = log_reg_over.predict(X_train_over)
y_pred_test_over = log_reg_over.predict(X_test)


# 5. Accuracy
acc_train_over = accuracy_score(
    y_train_over,
    y_pred_train_over
)

acc_test_over = accuracy_score(
    y_test,
    y_pred_test_over
)

precision_over = precision_score(
    y_test,
    y_pred_test_over
)

recall_over = recall_score(
    y_test,
    y_pred_test_over
)

f1_over = f1_score(
    y_test,
    y_pred_test_over
)

gap_over = acc_train_over - acc_test_over

In [48]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_test_over,
        zero_division=0
    )
)


Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.57      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [47]:
print(f"Accuracy (Train): {acc_train_over * 100:.2f}%")
print(f"Accuracy (Test): {acc_test_over * 100:.2f}%")
print(f"Precision (Test): {precision_over * 100:.2f}%")
print(f"Recall (Test): {recall_over * 100:.2f}%")
print(f"F1-score (Test): {f1_over * 100:.2f}%")
print(f"Train-Test Gap: {gap_over * 100:.2f}%")

Accuracy (Train): 94.07%
Accuracy (Test): 93.42%
Precision (Test): 57.49%
Recall (Test): 95.06%
F1-score (Test): 71.65%
Train-Test Gap: 0.64%


In [88]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": [
        "Original",
        "Oversampling"
    ],
    
    "Train Accuracy": [
        acc_train,
        acc_train_over
    ],
    
    "Test Accuracy": [
        acc_test,
        acc_test_over
 ],
    
    "Precision": [
        precision,
        precision_over
 ],
    
    "Recall": [
        recall,
        recall_over
],
    
    "F1-score": [
        f1,
        f1_over
    ],
    
    "Train-Test Gap": [
        gap,
        gap_over
    ]
})

# Convert to percentage
comparison[
    [
        "Train Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Train-Test Gap"
    ]
] *= 100

comparison.round(2)

,Method,Train Accuracy,Test Accuracy,Precision,Recall,F1-score,Train-Test Gap
0,Original,95.90,95.79,88.47,59.63,71.24,0.10
1,Oversampling,94.07,93.42,57.49,95.06,71.52,0.74


Original method is a little bit better because the precision is better regarding that the other recall is about a 95% it is only good for the true positives.


5. 

In [64]:
rus = RandomUnderSampler(random_state=SEED)

X_train_under, y_train_under = rus.fit_resample(
    X_train,
    y_train
)
print("Original:")
print(y_train.value_counts())

print("\nAfter Undersampling:")
print(y_train_under.value_counts())

Original:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After Undersampling:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


In [68]:
# 2. Create the Logistic Regression model

log_reg_under = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)

# 3. Train the model
log_reg_under.fit(X_train_under, y_train_under)


print ("Model trained!")

Model trained!


In [72]:
# 4. Predictions
y_pred_train_under = log_reg_under.predict(X_train_under)
y_pred_test_under = log_reg_under.predict(X_test)


# 5. Metrics
acc_train_under = accuracy_score(
    y_train_under,
    y_pred_train_under
)

acc_test_under = accuracy_score(
    y_test,
    y_pred_test_under
)

precision_under = precision_score(
    y_test,
    y_pred_test_under
)

recall_under = recall_score(
    y_test,
    y_pred_test_under
)

f1_under = f1_score(
    y_test,
    y_pred_test_under
)

gap_under = acc_train_under - acc_test_under


# 6. Results
print(f"Accuracy (Train): {acc_train_under * 100:.2f}%")
print(f"Accuracy (Test): {acc_test_under * 100:.2f}%")
print(f"Precision (Test): {precision_under * 100:.2f}%")
print(f"Recall (Test): {recall_under * 100:.2f}%")
print(f"F1-score (Test): {f1_under * 100:.2f}%")
print(f"Train-Test Gap: {gap_under * 100:.2f}%")


# 7. Classification Report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_test_under,
        zero_division=0
    )
)

Accuracy (Train): 94.12%
Accuracy (Test): 93.38%
Precision (Test): 57.33%
Recall (Test): 95.03%
F1-score (Test): 71.52%
Train-Test Gap: 0.74%

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.57      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [91]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": [
        "Original",
        "Oversampling",
        "Undersampling",
    ],
    
    "Train Accuracy": [
        acc_train,
        acc_train_over,
        acc_train_under
    ],
    
    "Test Accuracy": [
        acc_test,
        acc_test_over,
        acc_test_under
    ],
    
    "Precision": [
        precision,
        precision_over,
        precision_under
    ],
    
    "Recall": [
        recall,
        recall_over,
        recall_under
    ],
    
    "F1-score": [
        f1,
        f1_over,
        f1_under
    ],
    
    "Train-Test Gap": [
        gap,
        gap_over,
        gap_under
    ]
})

# Convert to percentage
comparison[
    [
        "Train Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Train-Test Gap"
    ]
] *= 100

comparison.round(2)

,Method,Train Accuracy,Test Accuracy,Precision,Recall,F1-score,Train-Test Gap
0,Original,95.90,95.79,88.47,59.63,71.24,0.10
1,Oversampling,94.07,93.42,57.49,95.06,71.52,0.74
2,Undersampling,94.12,93.38,57.33,95.03,71.52,0.74


Same than the other case there is no big difference in this case between under and over sampling

.6

In [76]:
smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)
print("Original:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Original:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

After SMOTE:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [77]:
log_reg_smote = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)

log_reg_smote.fit(
    X_train_smote,
    y_train_smote
)
print("Model trained!")

Model trained!


In [78]:
# 4. Predictions
y_pred_train_smote = log_reg_smote.predict(X_train_smote)
y_pred_test_smote = log_reg_smote.predict(X_test)

# 5. Metrics
acc_train_smote = accuracy_score(
    y_train_smote,
    y_pred_train_smote
)

acc_test_smote = accuracy_score(
    y_test,
    y_pred_test_smote
)

precision_smote = precision_score(
    y_test,
    y_pred_test_smote
)

recall_smote = recall_score(
    y_test,
    y_pred_test_smote
)

f1_smote = f1_score(
    y_test,
    y_pred_test_smote
)

gap_smote = acc_train_smote - acc_test_smote

# 6. Results
print(f"Accuracy (Train): {acc_train_smote * 100:.2f}%")
print(f"Accuracy (Test): {acc_test_smote * 100:.2f}%")
print(f"Precision (Test): {precision_smote * 100:.2f}%")
print(f"Recall (Test): {recall_smote * 100:.2f}%")
print(f"F1-score (Test): {f1_smote * 100:.2f}%")
print(f"Train-Test Gap: {gap_smote * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_test_smote,
        zero_division=0
    )
)

Accuracy (Train): 94.17%
Accuracy (Test): 93.47%
Precision (Test): 57.69%
Recall (Test): 94.82%
F1-score (Test): 71.74%
Train-Test Gap: 0.70%

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [90]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": [
        "Original",
        "Oversampling",
        "Undersampling",
        "SMOTE"
    ],
    
    "Train Accuracy": [
        acc_train,
        acc_train_over,
        acc_train_under,
        acc_train_smote
    ],
    
    "Test Accuracy": [
        acc_test,
        acc_test_over,
        acc_test_under,
        acc_test_smote
    ],
    
    "Precision": [
        precision,
        precision_over,
        precision_under,
        precision_smote
    ],
    
    "Recall": [
        recall,
        recall_over,
        recall_under,
        recall_smote
    ],
    
    "F1-score": [
        f1,
        f1_over,
        f1_under,
        f1_smote
    ],
    
    "Train-Test Gap": [
        gap,
        gap_over,
        gap_under,
        gap_smote
    ]
})

# Convert to percentage
comparison[
    [
        "Train Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Train-Test Gap"
    ]
] *= 100

comparison.round(2)

,Method,Train Accuracy,Test Accuracy,Precision,Recall,F1-score,Train-Test Gap
0,Original,95.90,95.79,88.47,59.63,71.24,0.10
1,Oversampling,94.07,93.42,57.49,95.06,71.52,0.74
2,Undersampling,94.12,93.38,57.33,95.03,71.52,0.74
3,SMOTE,94.17,93.47,57.69,94.82,71.74,0.70


The performance did not improve in terms of accuracy or precision. However, the resampling techniques substantially improved recall, which may be valuable when detecting positive cases is more important than avoiding false positives.